# Assistants API - Knowledge Retrieval 

https://platform.openai.com/docs/assistants/tools/knowledge-retrieval

https://community.openai.com/t/new-assistants-api-a-potential-replacement-for-low-level-rag-style-content-generation/475677 

Watch:

https://youtu.be/5rcjGjgJNQc?t=600&si=d9OtX0nMi2Rv0fQV 

References:

https://community.openai.com/t/assistants-api-retrieval-pricing-how-much-does-this-cost/485188/8

https://medium.com/madhukarkumar/what-does-openais-announcement-mean-for-retrieval-augmented-generation-rag-and-vector-only-54bfc34cba2c

https://www.youtube.com/watch?v=ClfyQNkTeUc

https://www.pinecone.io/learn/assistants-api-canopy/




![Alt text](ret.jpg "Assistants")

![Alt text](objects.jpeg "Assistants_Objects")

https://cobusgreyling.medium.com/openai-assistant-with-retriever-tool-08e9158ca900 

In [21]:
from openai import OpenAI
import json
from dotenv import load_dotenv, find_dotenv

_ : bool = load_dotenv(find_dotenv()) # read local .env file

In [22]:
client : OpenAI = OpenAI()

### Knowledge Retrieval

Retrieval augments the Assistant with knowledge from outside its model, such as proprietary product information or documents provided by your users. Once a file is uploaded and passed to the Assistant, OpenAI will automatically chunk your documents, index and store the embeddings, and implement vector search to retrieve relevant content to answer user queries.

https://platform.openai.com/docs/assistants/tools/knowledge-retrieval 



### How it works

The model then decides when to retrieve content based on the user Messages. The Assistants API automatically chooses between two retrieval techniques:

1. it either passes the file content in the prompt for short documents, or
2. performs a vector search for longer documents

Retrieval currently optimizes for quality by adding all relevant content to the context of model calls. We plan to introduce other retrieval strategies to enable developers to choose a different tradeoff between retrieval quality and model usage cost.

https://platform.openai.com/docs/assistants/tools/how-it-works


### Step 1: Upload the file and Create an Assistant

In [36]:
from openai.types.beta import Assistant

# Upload a file with an "assistants" purpose
file = client.files.create(
    file=open("zia_profile.pdf", "rb"),
    purpose="assistants"
)

# 2. Create a vector store (index + embeddings) for that file
vector_store = client.vector_stores.create(
    name="ZiaProfileIndex",
    file_ids=[file.id]            # <-- embed your uploaded PDF here
)



In [37]:
assistant = client.beta.assistants.create(
    name="Student Support Assistant",
    instructions=(
        "You are a student support chatbot. "
        "Use your knowledge base to best respond to student queries about Zia U. Khan."
    ),
    model="gpt-3.5-turbo-1106",
    tools=[{"type": "file_search"}],
    tool_resources={
        "file_search": {
            "vector_store_ids": [vector_store.id]
        }
    }
)


### Step 2: Create a Thread

In [38]:
from openai.types.beta.thread import Thread

thread: Thread  = client.beta.threads.create()

print(thread)


/tmp/ipykernel_88788/122329800.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  thread: Thread  = client.beta.threads.create()


Thread(id='thread_Gdyv7AVPNkKKu7U2eBlrFMnO', created_at=1751985875, metadata={}, object='thread', tool_resources=ToolResources(code_interpreter=None, file_search=None))


### Step 3: Add a Message to a Thread

In [39]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="When and which city Zia U. Khan was born?"
)


/tmp/ipykernel_88788/3962593845.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  message = client.beta.threads.messages.create(


### Step 4: Run the Assistant

In [40]:
from openai.types.beta.threads.run import Run

run: Run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id,
  instructions="Please address the user as Pakistani. The user is the student of PIAIC."
)


/tmp/ipykernel_88788/2007148430.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run: Run = client.beta.threads.runs.create(


### Step 5: Check the Run status

In [41]:
run: Run = client.beta.threads.runs.retrieve(
  thread_id=thread.id,
  run_id=run.id
)

print(run)


/tmp/ipykernel_88788/1771080927.py:1: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  run: Run = client.beta.threads.runs.retrieve(


Run(id='run_7x1Qf9dYDuuX7prNn4CoD5q8', assistant_id='asst_bdlPBNzBOrlJarJBppmiA7Aq', cancelled_at=None, completed_at=None, created_at=1751985878, expires_at=1751986478, failed_at=None, incomplete_details=None, instructions='Please address the user as Pakistani. The user is the student of PIAIC.', last_error=None, max_completion_tokens=None, max_prompt_tokens=None, metadata={}, model='gpt-3.5-turbo-1106', object='thread.run', parallel_tool_calls=True, required_action=None, response_format='auto', started_at=1751985879, status='in_progress', thread_id='thread_Gdyv7AVPNkKKu7U2eBlrFMnO', tool_choice='auto', tools=[FileSearchTool(type='file_search', file_search=FileSearch(max_num_results=None, ranking_options=FileSearchRankingOptions(score_threshold=0.0, ranker='default_2024_08_21')))], truncation_strategy=TruncationStrategy(type='auto', last_messages=None), usage=None, temperature=1.0, top_p=1.0, tool_resources={}, reasoning_effort=None)


### Step 6: Display the Assistant's Response

In [43]:
# from openai.resources.beta.threads.messages.messages import SyncCursorPage 

messages = client.beta.threads.messages.list(
  thread_id=thread.id
)

for m in reversed(messages.data):
  print(m.role + ": " + m.content[0].text.value)


/tmp/ipykernel_88788/3855163449.py:3: DeprecationWarning: The Assistants API is deprecated in favor of the Responses API
  messages = client.beta.threads.messages.list(


user: When and which city Zia U. Khan was born?
assistant: Zia U. Khan was born in Sialkot in 1961. Sialkot is an important city in Pakistan known for its sports and manufacturing industries. This information was found in the document "Zia Profile" .
